# MES: Multi-Agent Evidence Synthesis - demo

This notebook answers one clinical question with the six MES agents (Supervisor, LiteratureMiner, RWD-Analyst,
KG-Specialist, Statistician and SafeChecker) and displays the final evidence synthesis.
The real-world data are synthetic records in the MIMIC-IV table layout (`Data/synthetic_mimic`); the knowledge graph is
a small example subset of iBKH (`Data/KG/example_kg.csv`); literature is retrieved from PubMed and ClinicalTrials.gov.

## 1. Setup and LLM backend

In [1]:
import logging
import warnings

from dotenv import load_dotenv
from transformers.utils import logging as hf_logging

from KGSpecialist import KGSpecialist
from llm_zoo import create_llm
from mes_pipeline import run_med_synth, save_outputs
from rwd_preprocessor import MIMICPreprocessor

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(name)s: %(message)s")
for noisy in ("httpx", "httpx2", "sentence_transformers", "paperscraper"):
    logging.getLogger(noisy).setLevel(logging.WARNING)
hf_logging.disable_progress_bar()

load_dotenv()  # reads OPENAI_API_KEY (and optional organization / project) from .env

W1016 03:50:25.920000 38642 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


True

In [2]:
# Backbones evaluated in the manuscript:
#   "gpt-5.4-nano"                    (OpenAI API, requires OPENAI_API_KEY in .env)
#   "gemma4", "medgemma4b", "gemma3"  (local models served by Ollama, see README)
LLM_MODEL = "gpt-5.4-nano"
MAX_WORKERS = 8  # parallel LLM calls; for Ollama, match OLLAMA_NUM_PARALLEL (e.g., 1-4)

llm = create_llm(LLM_MODEL)
print(llm.invoke("Reply with one word: ready"))

ready


## 2. Data sources

In [3]:
preprocessor = MIMICPreprocessor(
    mimic_path="Data/synthetic_mimic",
    covariate_csv_path="Data/synthetic_mimic/icu_first_day_covariates.csv.gz",
    llm=llm,
    data_source="synthetic MIMIC-IV-formatted demo data",  # use "MIMIC-IV" when pointing to the real database
)
preprocessor.load_data()

# Embeds the KG triplets with embeddinggemma-300m-medical (downloaded from Hugging Face on first use).
kg = KGSpecialist("Data/KG/example_kg.csv")

rwd_preprocessor: Loading MIMIC-IV tables from Data/synthetic_mimic
rwd_preprocessor: Loaded 30000 patients and 32975 admissions.
KGSpecialist: Embedding 146 KG triplets with sentence-transformers/embeddinggemma-300m-medical.


## 3. Run MES

In [4]:
QUESTION = "Does metformin reduce mortality in ICU patients with type 2 diabetes?"

state = run_med_synth(QUESTION, llm, preprocessor, kg, max_rounds=2, max_workers=MAX_WORKERS)

MES: ===== Round 1: Does metformin reduce mortality in ICU patients with type 2 diabetes?
Supervisor: Query remains ambiguous; continuing with the partial decomposition.
MES: RWD task: trial_emulation
rwd_preprocessor: Analysis type: time_to_event | outcome: OutcomeSpecification(outcome_type='mortality', follow_up_end=30) | treatment: TreatmentSpecification(treatment_medications=['metformin'], control_medications=None, treatment_window_days=None)
rwd_preprocessor: Selected 7809 patients meeting eligibility criteria.
rwd_preprocessor: Final dataframe: 7809 patients x 52 columns.
KGSpecialist: KG retrieval mode: structured; mapped nodes: {'metformin': 'Metformin', 'type 2 diabetes mellitus': 'type 2 diabetes mellitus'}
LiteratureMiner: Literature source decision: Both
LiteratureMiner: ClinicalTrials.gov: 20 trials found.
paperscraper.load_dumps:  No dump found for biorxiv. Skipping entry.
paperscraper.load_dumps:  No dump found for chemrxiv. Skipping entry.
paperscraper.load_dumps:  No d

## 4. Final evidence synthesis

In [5]:
from IPython.display import Markdown, display

display(Markdown(state["final_report"]))

Bottom line. In ICU patients with type 2 diabetes, metformin has **uncertain effect on mortality**: the most ICU-relevant causal estimate from a **synthetic-data target trial emulation** suggests a **modest reduction** in **30-day mortality** (HR **0.86**, 95% CI **0.74–1.00**; p=**0.0549**) and **28-day mortality** (HR **0.87**, 95% CI **0.75–1.02**; p=**0.0912**), but confidence intervals include no effect. Published ICU observational evidence is **inconsistent** (some cohorts show large mortality reductions; one frail elderly CKD cohort shows harm signals including lactic acidosis and higher composite adverse outcomes). Overall evidence is **low certainty** and **not fully consistent** across designs and populations.

Evidence from the published literature.  
### Randomized/prospective ICU trials with mortality results
- **No usable ICU trial mortality results were provided** for registered studies in ICU populations with type 2 diabetes (mortality endpoints were described, but no effect estimates were reported).  
  - ClinicalTrials.gov records: NCT05979038; NCT06181422; NCT06296836; NCT07591402. [1–4]

### Other published studies (non-randomized ICU evidence)
- **Acute respiratory failure ICU cohort (propensity score matched; observational; high risk of bias)**  
  - In-hospital mortality: **OR 0.202** (95% CI **0.123–0.317**; p<0.001)  
  - ICU mortality: **OR 0.245** (95% CI **0.142–0.399**; p<0.001)  
  - 30-day mortality: **HR 0.199** (95% CI **0.124–0.320**; p<0.001)  
  - 90-day mortality: **HR 0.230** (95% CI **0.150–0.352**; p<0.001)  
  - Source limitation: retrospective design; residual confounding/selection and exposure/misclassification concerns despite matching. [5]

- **Very elderly ICU patients with CKD (single-center retrospective observational; high risk of bias; harm signals)**  
  - All-cause mortality (longer follow-up; median 33.7 months): **HR 1.57** (95% CI **0.89–2.78**) — numerically higher, not statistically significant  
  - Primary composite adverse outcome (ICU admission, RRT initiation, lactic acidosis, or all-cause mortality): **HR 1.75** (95% CI **1.12–2.73**; p=0.013)  
  - ICU admission: **HR 2.33** (95% CI **1.33–4.08**)  
  - RRT initiation: **HR 1.90** (95% CI **1.14–3.16**)  
  - Lactic acidosis: **HR 3.14** (95% CI **1.75–5.65**)  
  - Source limitation: non-randomized; confounding/selection bias likely. [6]

### Safety-related context from ICU populations (not mortality effect in ICU T2D specifically)
- In **metformin-associated lactic acidosis (MALA)** patients admitted to ICU (n=91), outcomes differed significantly only for **vasoactive support** across contraindication categories (P=0.0149); other outcome distributions were not significantly different. This does not directly quantify metformin’s mortality effect in ICU T2D, but supports clinical plausibility of harm in susceptible contexts. [7]

Real-world evidence (synthetic MIMIC-IV-formatted demo data).  
- **Design:** ICU **target trial emulation** using synthetic MIMIC-IV demo data; **clone-censor-weight** approach with a **2-day grace period** to mitigate **immortal-time bias**, plus **IPTW** adjusting for baseline covariates from the **first ICU day** (SOFA components, labs/vitals, renal function markers, etc.); outcome was **all-cause mortality** from time zero.  
- **30-day mortality (primary):** HR **0.8601** (95% CI **0.7375–1.0031**; p=**0.0549**); weighted 30-day risk **0.1279** vs **0.1507** (risk difference **-0.0228**). [8]  
- **28-day mortality:** HR **0.8734** (95% CI **0.7465–1.0219**; p=**0.0912**); weighted 28-day risk **0.1236** vs **0.1434** (risk difference **-0.0198**). [9]  
- **Subgroups (exploratory):**  
  - 28-day mortality: renal <2 HR **0.8348** (95% CI **0.7309–0.9535**; p=0.0078) vs renal ≥2 HR **1.0069** (95% CI **0.7895–1.2842**; p=0.9556); interaction p=0.3878. [9]  
  - 28-day mortality: lactate <2 HR **0.7826** (95% CI **0.6181–0.9908**; p=0.0416) vs lactate ≥2 HR **0.9139** (95% CI **0.7991–1.0451**; p=0.1883); interaction p=0.2996. [9]  
- **Key limitations:** synthetic-data setting; residual confounding and operational deviations from an ideal target trial (e.g., time-zero and initiation window definitions); causal assumptions unverifiable. [8,9]

Knowledge-graph context (iBKH).  
- iBKH provides **context only**: metformin is linked to treating type 2 diabetes, and a 2-hop path suggests metformin **downregulates IL-6** while type 2 diabetes is associated with **IL-6**—a plausible anti-inflammatory mechanism, but **no direct clinical evidence** for ICU mortality effect. [10]

Agreement and divergence across sources.  
- **Direction of effect is not consistent across ICU evidence classes.**  
  - Synthetic target emulation (30-day/28-day): **HR <1** but **CIs include 1** (borderline/uncertain benefit). [8,9]  
  - Observational ARF ICU cohort: **substantial mortality reductions** (ORs and HRs far below 1, all p<0.001). [5]  
  - Observational very elderly CKD ICU cohort: **harm signals** for composite adverse outcomes and for lactic acidosis/RRT initiation; all-cause mortality HR **>1** but not statistically significant. [6]  
- Plausible explanations for divergence include **population differences** (frail elderly CKD vs broader ARF ICU vs synthetic emulation cohort), **exposure definition** (metformin use vs early initiation strategy), **outcome horizon** (in-hospital/ICU vs 28–30 days vs longer follow-up), and **bias handling** (propensity matching vs target trial emulation vs residual confounding).

Limitations.  
- **No randomized ICU trial mortality results** were available in the provided evidence set; registered ICU trials lacked reported mortality effect estimates. [1–4]  
- All mortality estimates come from **non-randomized designs** (retrospective cohorts) or **synthetic-data emulation**, leaving **residual confounding/selection bias** and **measurement error** as major threats.  
- **Estimand mismatch:** exposure definitions differ (metformin use vs initiation within a time window), and mortality horizons differ (in-hospital/ICU, 28-day, 30-day, longer follow-up).  
- **Synthetic-data limitations**: external validity and fidelity of treatment assignment/measurement cannot be guaranteed. [8,9]  
- Safety signals (lactic acidosis, RRT initiation) appear in a high-risk CKD subgroup, but mortality effects in that subgroup are imprecise. [6]

References:  
[1] https://clinicaltrials.gov/study/NCT05979038  
[2] https://clinicaltrials.gov/study/NCT06181422  
[3] https://clinicaltrials.gov/study/NCT06296836  
[4] https://clinicaltrials.gov/study/NCT07591402  
[5] Correlation between metformin use and mortality in acute respiratory failure: a retrospective ICU cohort study. doi:10.3389/fphar.2025.1584230  
[6] Metformin Use and Clinical Outcomes in Very Elderly Patients with Type 2 Diabetes and Chronic Kidney Disease. doi:10.3390/medicina62040776  
[7] Metformin use despite contraindications in metformin-associated lactic acidosis. doi:10.1097/MD.0000000000045197  
[8] Real-world data analysis on synthetic MIMIC-IV-formatted demo data  
[9] Real-world data analysis on synthetic MIMIC-IV-formatted demo data  
[10] Knowledge graph relations from iBKH

In [6]:
save_outputs(state, "outputs/demo")  # final_report.md and agent_outputs.json

## 5. Intermediate agent outputs

Knowledge-graph relations passed to the Statistician and the real-world estimate of the RWD-Analyst. The synthetic data
were simulated with a known treatment effect (`Data/synthetic_mimic/ground_truth.json`), so the real-world estimate can
be compared with the truth.

In [7]:
import json

import pandas as pd

if state["kg"]:
    print("KG retrieval:", state["kg"]["retrieval_info"])
    display(pd.DataFrame(state["kg"]["evidence"])[["text", "retrieval", "similarity", "reliability"]])

for rwd in state["rwd"]:
    print(f"RWD-Analyst ({rwd.get('task_type')}): {rwd['query']}")
    print(rwd["structured_result"] or rwd["error"])

with open("Data/synthetic_mimic/ground_truth.json") as f:
    print("Simulated truth:", json.load(f))

KG retrieval: {'entities': ['metformin', 'type 2 diabetes mellitus', 'mortality'], 'mapped_nodes': {'metformin': 'Metformin', 'type 2 diabetes mellitus': 'type 2 diabetes mellitus'}, 'mode': 'structured', 'n_paths': 19}


,text,retrieval,similarity,reliability
0,Metformin treats type 2 diabetes mellitus,structured: 1-hop,0.6118,High
1,Metformin has an inferred relation with diabet...,structured: 1-hop,0.5888,Moderate
2,"Metformin has an inferred relation with Death,...",structured: 1-hop,0.5879,Moderate
3,"Metformin has an inferred relation with Death,...",structured: 1-hop,0.5851,Moderate
4,Metformin has an inferred relation with Critic...,structured: 1-hop,0.5840,Low
5,Metformin is associated with kidney failure,structured: 1-hop,0.5789,Low
6,Metformin is associated with kidney disease,structured: 1-hop,0.5752,Low
7,Metformin downregulates IL6; type 2 diabetes m...,structured: 2-hop path via IL6,0.4846,Moderate
8,Metformin is associated with SLC2A2; type 2 di...,structured: 2-hop path via SLC2A2,0.4823,Moderate
9,Metformin has an inferred relation with diabet...,structured: 2-hop path via diabetic ketoacidosis,0.4819,Moderate


RWD-Analyst (trial_emulation): Does metformin reduce mortality in ICU patients with type 2 diabetes?
{'effect_measure': 'HR', 'effect_size': 0.8601019032106563, 'ci_lower': 0.7374567034722201, 'ci_upper': 1.0031440224537334, 'p_value': 0.05486143410004644, 'risk_treated': 0.12789801594174866, 'risk_control': 0.15071423570739806, 'method': 'clone-censor-weight (grace period 2 days) + weighted Cox', 'n_patients': 7809, 'n_treated': 3060, 'outcome': '30-day mortality', 'treatment': 'initiate metformin within 2 days after time zero vs do not initiate metformin within 2 days'}
RWD-Analyst (trial_emulation): In ICU patients with type 2 diabetes, what is the effect of metformin on mortality with a clearly specified time horizon (e.g., in-hospital mortality or 28-day mortality), and are there any ICU-focused randomized or prospective studies reporting that mortality endpoint?
{'effect_measure': 'HR', 'effect_size': 0.8734185508706273, 'ci_lower': 0.7464807092714677, 'ci_upper': 1.0219419678633